# 🦜 LangServe + 🔭 LangSmith — Complete Revision Notebook
### *Production-Grade | Interview-Ready | Industry Best Practices*

---

```
Author     : Rakesh Kumar (SDE2 @ CGI | Masters AI/ML)
Purpose    : Deep revision of LangServe & LangSmith for ML/GenAI interviews
Stack      : LangServe (FastAPI-based deployment) + LangSmith (Observability)
```

---

## 🗺️ The LangChain Ecosystem — Where These Fit

```
┌──────────────────────────────────────────────────────────────────┐
│                   LANGCHAIN ECOSYSTEM                            │
│                                                                  │
│  🧱 LangChain Core    →  Build chains, agents, prompts           │
│  🔗 LangGraph         →  Stateful, cyclic, multi-agent flows     │
│  🚀 LangServe         →  Deploy chains/agents as REST APIs       │
│  🔭 LangSmith         →  Observe, debug, evaluate, test          │
│                                                                  │
│  Think of it as:                                                 │
│    Build (LangChain) → Orchestrate (LangGraph)                  │
│    → Deploy (LangServe) → Monitor (LangSmith)                   │
└──────────────────────────────────────────────────────────────────┘
```

## 📋 Table of Contents

| # | Section | Tool |
|---|---------|------|
| **PART 1** | **LANGSERVE** | 🚀 |
| 1 | What is LangServe & Why it matters | LangServe |
| 2 | Installation & Project Setup | LangServe |
| 3 | Core Concepts: add_routes, Runnable | LangServe |
| 4 | Deploying a Simple Chain | LangServe |
| 5 | Deploying a RAG Chain | LangServe |
| 6 | Deploying a LangGraph Agent | LangServe |
| 7 | Streaming Endpoints | LangServe |
| 8 | Input/Output Schemas & Validation | LangServe |
| 9 | RemoteRunnable — Client Side | LangServe |
| 10 | Auth, CORS, Custom Middleware | LangServe |
| 11 | LangServe Playground | LangServe |
| **PART 2** | **LANGSMITH** | 🔭 |
| 12 | What is LangSmith & Why it matters | LangSmith |
| 13 | Setup — Step by Step | LangSmith |
| 14 | Tracing — Auto vs Manual | LangSmith |
| 15 | Datasets & Evaluation | LangSmith |
| 16 | Custom Evaluators | LangSmith |
| 17 | Feedback & Annotation | LangSmith |
| 18 | Monitoring in Production | LangSmith |
| 19 | LangSmith Hub — Prompt Management | LangSmith |
| **PART 3** | **Interview Q&A + Industry Patterns** | 🔥 |
| 20 | Interview Questions with Answers | Both |
| 21 | Industry Patterns & System Design | Both |


---
# 🚀 PART 1: LANGSERVE
### *Deploy your LangChain apps as production REST APIs in minutes*


---
## 1. 🧠 What is LangServe & Why It Matters

### Theory
LangServe is a **deployment library** that wraps any LangChain `Runnable` (chain, agent, prompt, LLM) and exposes it as a **production-ready FastAPI REST API** — automatically, with zero boilerplate.

### What you get for FREE with LangServe

```
Your LangChain Runnable
        │
        ▼  add_routes(app, chain, path="/chat")
        │
        ▼
┌───────────────────────────────────────────┐
│  AUTO-GENERATED ENDPOINTS                 │
│                                           │
│  POST /chat/invoke      → sync call       │
│  POST /chat/batch       → batch calls     │
│  POST /chat/stream      → streaming SSE   │
│  POST /chat/astream_log → detailed stream │
│  GET  /chat/input_schema  → JSON schema   │
│  GET  /chat/output_schema → JSON schema   │
│  GET  /chat/config_schema → JSON schema   │
│  GET  /chat/playground    → Interactive UI│
└───────────────────────────────────────────┘
```

### Why LangServe over plain FastAPI?

| Feature | Plain FastAPI | LangServe |
|---------|--------------|----------|
| Endpoint setup | Manual for each | Auto-generated |
| Streaming | Manual SSE setup | Built-in |
| Schema validation | Manual Pydantic | Auto from Runnable |
| Playground UI | None | Built-in |
| Batch endpoint | Manual | Built-in |
| LangSmith integration | Manual | Automatic |
| Client SDK | None | `RemoteRunnable` |

> 💡 **Interview Tip**: LangServe = FastAPI + LangChain Runnable Protocol + Auto schema + Streaming. Mention this when asked about MLOps or LLM deployment.

### Current Status (2024-2025)
> ⚠️ **Important**: LangServe is in **maintenance mode**. LangChain team recommends **LangGraph Platform / LangGraph Cloud** for new production deployments of agents. LangServe is still widely used for **simple chain deployments** and in existing production systems.


---
## 2. ⚙️ Installation & Project Setup


In [ ]:
# Install LangServe with all optional dependencies
!pip install "langserve[all]"  # includes client + server deps
!pip install langchain langchain-openai
!pip install uvicorn  # ASGI server to run FastAPI

In [ ]:
# ── Production Project Structure ──────────────────────
PROJECT_STRUCTURE = """
my_langserve_app/
├── app/
│   ├── __init__.py
│   ├── server.py          ← Main FastAPI app + add_routes
│   ├── chains/
│   │   ├── chat_chain.py
│   │   ├── rag_chain.py
│   │   └── agent.py
│   └── config.py          ← Settings, API keys
├── tests/
│   └── test_chains.py
├── .env                   ← API keys
├── Dockerfile
├── docker-compose.yml
└── requirements.txt
"""
print(PROJECT_STRUCTURE)

---
## 3. 🧩 Core Concepts: add_routes & Runnable Protocol

### The Runnable Protocol
Everything in LangChain implements `Runnable`. This is the contract LangServe depends on:

```python
# Any Runnable has these methods:
runnable.invoke(input)          # sync, single
runnable.batch([input1, input2]) # sync, multiple
runnable.stream(input)          # sync, streaming
await runnable.ainvoke(input)   # async, single
await runnable.abatch([...])    # async, multiple  
await runnable.astream(input)   # async, streaming
```

### add_routes — The Magic Function
```python
add_routes(
    app,              # FastAPI app
    runnable,         # Any LangChain Runnable
    path="/chat",     # URL prefix
    input_type=...,   # Optional Pydantic model
    output_type=...,  # Optional Pydantic model
    config_keys=[],   # Expose config keys to client
    enable_feedback_endpoint=True,  # LangSmith feedback
)
```


---
## 4. 🌱 Deploying a Simple Chain


In [ ]:
# ── server.py ─────────────────────────────────────────
# Save this file and run: uvicorn server:app --reload --port 8000

SIMPLE_SERVER_CODE = '''
from fastapi import FastAPI
from langserve import add_routes
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

# ── FastAPI App ───────────────────────────────────────
app = FastAPI(
    title="LangServe Demo",
    version="1.0",
    description="Production LangChain API"
)

# ── Chain 1: Simple Q&A ───────────────────────────────
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer concisely."),
    ("human", "{question}")
])

qa_chain = qa_prompt | llm | StrOutputParser()

# ── Chain 2: Translation ──────────────────────────────
translate_prompt = ChatPromptTemplate.from_template(
    "Translate the following to {language}: {text}"
)
translate_chain = translate_prompt | llm | StrOutputParser()

# ── Add Routes (auto-generates ALL endpoints) ─────────
add_routes(app, qa_chain, path="/qa")
add_routes(app, translate_chain, path="/translate")

# ── Health Check ──────────────────────────────────────
@app.get("/health")
def health(): return {"status": "ok", "service": "langserve-demo"}

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

print(SIMPLE_SERVER_CODE)
print("\n" + "="*60)
print("After running, these endpoints are available:")
endpoints = [
    "POST /qa/invoke         → single Q&A call",
    "POST /qa/batch          → multiple Q&A calls",
    "POST /qa/stream         → streaming Q&A",
    "GET  /qa/playground     → interactive UI",
    "GET  /qa/input_schema   → input JSON schema",
    "POST /translate/invoke  → single translation",
    "GET  /docs              → Swagger UI",
]
for e in endpoints:
    print(f"  {e}")

---
## 5. 📚 Deploying a RAG Chain


In [ ]:
RAG_SERVER_CODE = '''
# ── rag_chain.py ──────────────────────────────────────
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.documents import Document

# ── Build vector store (in production: load from disk/cloud) ──
docs = [
    Document(page_content="LangChain is a framework for LLM applications"),
    Document(page_content="LangServe deploys LangChain apps as APIs"),
    Document(page_content="LangSmith provides observability for LLM apps"),
]
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# ── RAG Prompt ────────────────────────────────────────
rag_prompt = ChatPromptTemplate.from_template("""
Answer the question based on the context below.
If you cannot answer from context, say "I don't know".

Context: {context}

Question: {question}

Answer:
""")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ── RAG Chain using LCEL ──────────────────────────────
def format_docs(docs):
    return "\\n\\n".join(doc.page_content for doc in docs)

# RunnableParallel runs retrieval + passthrough simultaneously
rag_chain = (
    RunnableParallel(
        context=(lambda x: x["question"]) | retriever | format_docs,
        question=RunnablePassthrough() | (lambda x: x["question"])
    )
    | rag_prompt
    | llm
    | StrOutputParser()
)

# ── In server.py ──────────────────────────────────────
# from rag_chain import rag_chain
# add_routes(app, rag_chain, path="/rag")
'''

print(RAG_SERVER_CODE)

---
## 6. 🤖 Deploying a LangGraph Agent via LangServe

> 🔥 **Most asked combo**: LangGraph (orchestration) + LangServe (deployment) — knowing this is a big differentiator!


In [ ]:
AGENT_SERVER_CODE = '''
# ── Deploying a LangGraph agent via LangServe ─────────
from fastapi import FastAPI
from langserve import add_routes
from langgraph.graph import StateGraph, END, START
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
from operator import add
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# ── Build your LangGraph agent ────────────────────────
class AgentState(TypedDict):
    messages: Annotated[list, add]

llm = ChatOpenAI(model="gpt-4o-mini")

def agent_node(state: AgentState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

workflow = StateGraph(AgentState)
workflow.add_node("agent", agent_node)
workflow.add_edge(START, "agent")
workflow.add_edge("agent", END)

memory = MemorySaver()
agent_app = workflow.compile(checkpointer=memory)

# ── FastAPI + LangServe ───────────────────────────────
app = FastAPI(title="LangGraph Agent API")

# LangGraph compiled app IS a Runnable — plug directly!
add_routes(
    app,
    agent_app,
    path="/agent",
    config_keys=["configurable"],  # expose thread_id to client
)

# Client call with thread_id (enables memory):
# POST /agent/invoke
# Body: {
#   "input": {"messages": [{"role": "human", "content": "Hello"}]},
#   "config": {"configurable": {"thread_id": "user_123"}}
# }
'''

print(AGENT_SERVER_CODE)

---
## 7. 📡 Streaming Endpoints

### Streaming in LangServe — 3 Modes

```
POST /chain/stream       → streams final output tokens
POST /chain/stream_log   → streams intermediate steps + tokens  
POST /chain/astream_log  → async version with full trace
```


In [ ]:
import requests
import json

# ── Client-side streaming (how frontend would call it) ─
def stream_from_langserve(question: str, server_url: str = "http://localhost:8000"):
    """
    Consume a LangServe streaming endpoint.
    Returns tokens as they arrive — use this in FastAPI SSE or WebSocket.
    """
    url = f"{server_url}/qa/stream"
    payload = {"input": {"question": question}}
    
    with requests.post(url, json=payload, stream=True) as response:
        for line in response.iter_lines():
            if line:
                decoded = line.decode("utf-8")
                if decoded.startswith("data: "):
                    data = decoded[6:]  # strip "data: "
                    if data != "[DONE]":
                        try:
                            chunk = json.loads(data)
                            print(chunk, end="", flush=True)
                        except json.JSONDecodeError:
                            pass

# ── Production: AsyncIO version for FastAPI ───────────
ASYNC_STREAM_CODE = '''
import httpx
from fastapi import FastAPI
from fastapi.responses import StreamingResponse

async def proxy_stream(question: str):
    """Proxy streaming from LangServe to your frontend"""
    async with httpx.AsyncClient() as client:
        async with client.stream(
            "POST",
            "http://localhost:8000/qa/stream",
            json={"input": {"question": question}},
            timeout=60
        ) as response:
            async for chunk in response.aiter_text():
                yield chunk

@app.get("/stream")
async def stream_endpoint(question: str):
    return StreamingResponse(proxy_stream(question), media_type="text/event-stream")
'''

print("✅ Streaming client patterns loaded")
print("\nKey streaming endpoint types:")
streaming_types = [
    ("/stream",        "Final output tokens only",         "Best for chatbots"),
    ("/stream_log",    "All intermediate steps + tokens",  "Best for debugging"),
    ("/astream_events","Full event stream (LangGraph)",    "Best for complex agents"),
]
for endpoint, desc, use_case in streaming_types:
    print(f"  {endpoint:<20} | {desc:<40} | {use_case}")

---
## 8. 📐 Input/Output Schemas & Validation

### Why Schema Matters in Production
- Auto-generates OpenAPI/Swagger documentation
- Validates client requests before they hit your LLM (saves cost)
- Enables type-safe `RemoteRunnable` client calls


In [ ]:
from pydantic import BaseModel, Field
from typing import Optional, List

# ── Define strict input/output schemas ───────────────
class RAGInput(BaseModel):
    question: str = Field(..., description="The user's question", min_length=3, max_length=500)
    language: Optional[str] = Field("english", description="Response language")
    top_k: Optional[int] = Field(3, ge=1, le=10, description="Number of docs to retrieve")

    class Config:
        json_schema_extra = {
            "example": {
                "question": "What is LangServe?",
                "language": "english",
                "top_k": 3
            }
        }

class RAGOutput(BaseModel):
    answer: str
    sources: Optional[List[str]] = []
    confidence: Optional[float] = None

# ── Usage in add_routes ───────────────────────────────
SCHEMA_USAGE = '''
# In server.py:
add_routes(
    app,
    rag_chain,
    path="/rag",
    input_type=RAGInput,      # validates all incoming requests
    output_type=RAGOutput,    # documents the response format
)

# Now GET /rag/input_schema returns:
# {
#   "properties": {
#     "question": {"type": "string", "minLength": 3, ...},
#     "language": {"type": "string", "default": "english"},
#     ...
#   }
# }
'''

print("Input Schema:")
import json
print(json.dumps(RAGInput.model_json_schema(), indent=2))

---
## 9. 🖥️ RemoteRunnable — Client-Side SDK

### Theory
`RemoteRunnable` is LangServe's **client SDK**. It wraps a LangServe endpoint and makes it behave exactly like a local Runnable — same `.invoke()`, `.stream()`, `.batch()` interface.

```
Your Python App / Jupyter
        │
        │  from langserve import RemoteRunnable
        │  chain = RemoteRunnable("http://server:8000/qa")
        │
        ▼
LangServe API (running on another machine/cloud)
```

**Use case**: Microservices architecture — one service per chain/agent, all called via `RemoteRunnable`.


In [ ]:
from langserve import RemoteRunnable

# ── Connect to a remote LangServe endpoint ────────────
# (Server must be running at this URL)
qa_chain = RemoteRunnable("http://localhost:8000/qa")

# ── Use EXACTLY like a local chain ────────────────────
# Single call
# result = qa_chain.invoke({"question": "What is LangServe?"})

# Batch call
# results = qa_chain.batch([
#     {"question": "What is LangServe?"},
#     {"question": "What is LangSmith?"},
#     {"question": "What is LangGraph?"},
# ])

# Streaming
# for token in qa_chain.stream({"question": "Explain RAG in detail"}):
#     print(token, end="", flush=True)

# Async
# result = await qa_chain.ainvoke({"question": "Hello!"})

# ── With config (for stateful agents) ────────────────
agent = RemoteRunnable("http://localhost:8000/agent")
# result = agent.invoke(
#     {"messages": [{"role": "human", "content": "Hello"}]},
#     config={"configurable": {"thread_id": "user_42"}}
# )

print("✅ RemoteRunnable interface:")
print("  .invoke(input, config)        → sync single call")
print("  .batch([inputs], config)      → sync multiple calls")
print("  .stream(input, config)        → sync streaming")
print("  await .ainvoke(input, config) → async single call")
print("  await .astream(input, config) → async streaming")

---
## 10. 🔒 Auth, CORS & Custom Middleware

### Production Security Checklist


In [ ]:
PRODUCTION_SERVER_CODE = '''
from fastapi import FastAPI, Request, HTTPException, Depends
from fastapi.middleware.cors import CORSMiddleware
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from langserve import add_routes
import time

app = FastAPI(title="Production LangServe")

# ── CORS (required for browser clients) ──────────────
app.add_middleware(
    CORSMiddleware,
    allow_origins=["https://myapp.com", "http://localhost:3000"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── Request logging middleware ────────────────────────
@app.middleware("http")
async def log_requests(request: Request, call_next):
    start = time.time()
    response = await call_next(request)
    duration = time.time() - start
    print(f"{request.method} {request.url.path} → {response.status_code} ({duration:.3f}s)")
    return response

# ── API Key Authentication ────────────────────────────
security = HTTPBearer()
VALID_TOKENS = {"prod-token-abc123", "dev-token-xyz789"}

def verify_token(credentials: HTTPAuthorizationCredentials = Depends(security)):
    if credentials.credentials not in VALID_TOKENS:
        raise HTTPException(status_code=401, detail="Invalid API token")
    return credentials.credentials

# ── Protected routes (per-route auth) ────────────────
add_routes(
    app,
    qa_chain,
    path="/qa",
    dependencies=[Depends(verify_token)],  # require auth
)

# ── Rate limiting (use slowapi in production) ─────────
# from slowapi import Limiter
# limiter = Limiter(key_func=get_remote_address)
# @app.get("/")
# @limiter.limit("10/minute")
# async def root(request: Request): ...
'''

print(PRODUCTION_SERVER_CODE)
print("\n🔒 Production Security Checklist:")
checklist = [
    "✅ CORS configured for specific origins (not *)",
    "✅ API key / JWT authentication on all routes",
    "✅ Request logging middleware",
    "✅ Rate limiting (slowapi / nginx)",
    "✅ Input validation via Pydantic schemas",
    "✅ HTTPS only in production",
    "✅ Secrets in env vars, never in code",
]
for item in checklist:
    print(f"  {item}")

---
## 11. 🎮 LangServe Playground

### What is it?
Every LangServe endpoint automatically gets a **built-in interactive UI** at `GET /chain/playground`. It's like Swagger UI but specifically for LLM chains:

```
http://localhost:8000/qa/playground
```

**Features:**
- Input fields auto-generated from schema
- Real-time streaming output
- Shows intermediate steps
- Useful for:
  - Demo to stakeholders
  - Manual testing
  - Debugging prompts

> 💡 **Interview Tip**: The Playground is a free testing UI for any LangServe deployment. In demos, show this — it impresses non-technical stakeholders instantly.

### Disable in production (security)
```python
add_routes(app, chain, path="/qa", playground_type="default")   # enabled
add_routes(app, chain, path="/qa", disabled_endpoints=["playground"])  # disabled
```


---
---
# 🔭 PART 2: LANGSMITH
### *Observe, Debug, Evaluate, and Improve your LLM applications*


---
## 12. 🧠 What is LangSmith & Why It Matters

### Theory
LangSmith is an **observability + evaluation platform** for LLM applications. It answers the critical production questions:

```
❓ Why did my agent give a wrong answer?
❓ How many tokens did that chain consume?
❓ Which prompt version performs better?
❓ Is my RAG pipeline actually retrieving the right docs?
❓ How do I compare GPT-4 vs Claude on my use case?
❓ Which user inputs cause failures?
```

### LangSmith = 4 Core Capabilities

```
┌─────────────────────────────────────────────────────────────┐
│                     LANGSMITH PILLARS                       │
│                                                             │
│  1. TRACING       → Full execution trace of every run      │
│     (What happened, in what order, with what inputs/outputs)│
│                                                             │
│  2. EVALUATION    → Measure quality of LLM outputs         │
│     (Automated + human feedback on datasets)               │
│                                                             │
│  3. DATASETS      → Curate test cases & golden answers     │
│     (Version-controlled test sets for regression testing)  │
│                                                             │
│  4. HUB           → Prompt version management              │
│     (Store, version, share, pull prompts programmatically) │
└─────────────────────────────────────────────────────────────┘
```

### Why every production LLM app NEEDS LangSmith

| Without LangSmith | With LangSmith |
|-------------------|---------------|
| `print()` debugging | Full trace visualization |
| "It seems wrong" | Exact token-level trace |
| Manual A/B testing | Automated eval on datasets |
| No cost visibility | Token cost per run |
| Prompt in codebase | Versioned in Hub |
| No regression testing | CI/CD eval pipeline |

> 🔥 **Interview Tip**: When asked "How do you ensure LLM quality in production?" — answer with LangSmith tracing + datasets + eval pipeline. This alone sets you apart.


---
## 13. ⚙️ Setup — Step by Step

### Complete Setup Guide


In [ ]:
# ── STEP 1: Install ───────────────────────────────────
!pip install langsmith langchain langchain-openai

In [ ]:
# ── STEP 2: Get API Key ───────────────────────────────
# Go to: https://smith.langchain.com
# → Sign up / Log in
# → Settings → API Keys → Create Key

# ── STEP 3: Set Environment Variables ────────────────
import os

# These 4 env vars enable AUTOMATIC tracing of ALL LangChain calls
os.environ["LANGCHAIN_TRACING_V2"] = "true"          # Enable tracing
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"  # API URL
os.environ["LANGCHAIN_API_KEY"] = "ls__your_key_here"  # Your API key
os.environ["LANGCHAIN_PROJECT"] = "my-production-app"   # Project name (organizes runs)

# ── OR use .env file (recommended) ────────────────────
ENV_CONTENT = """
# .env file
LANGCHAIN_TRACING_V2=true
LANGCHAIN_ENDPOINT=https://api.smith.langchain.com
LANGCHAIN_API_KEY=ls__your_key_here
LANGCHAIN_PROJECT=my-production-app
OPENAI_API_KEY=sk-your-openai-key
"""
print("✅ Add to your .env file:")
print(ENV_CONTENT)

In [ ]:
# ── STEP 4: Verify Connection ─────────────────────────
from langsmith import Client

client = Client()

# Test connection
try:
    projects = list(client.list_projects())
    print("✅ LangSmith connected!")
    print(f"Projects: {[p.name for p in projects]}")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("Check your LANGCHAIN_API_KEY")

In [ ]:
# ── STEP 5: First Trace (automatic — just run any chain!) ─
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini")
prompt = ChatPromptTemplate.from_template("Answer briefly: {question}")
chain = prompt | llm | StrOutputParser()

# This call is AUTOMATICALLY traced in LangSmith!
# result = chain.invoke({"question": "What is LangSmith?"})
# print(result)

# Now go to https://smith.langchain.com → your project → see the trace!
print("✅ Steps 1-5 complete!")
print("\nWhat you'll see in LangSmith UI:")
trace_info = [
    "📊 Full execution trace (Prompt → LLM → Parser)",
    "⏱️  Latency for each step",
    "🔤 Exact prompt sent to LLM",
    "💬 Exact response from LLM",
    "💰 Token count + estimated cost",
    "❌ Errors with full stack trace",
]
for info in trace_info:
    print(f"  {info}")

---
## 14. 🔍 Tracing — Auto vs Manual

### Auto Tracing
Just set the 4 env vars → every LangChain/LangGraph call is traced automatically. Zero code change.

### Manual Tracing — When you need it
For **non-LangChain code** (raw OpenAI calls, custom functions, external APIs) you want to trace.


In [ ]:
from langsmith import traceable, Client
from langsmith.run_helpers import get_current_run_tree

# ── Method 1: @traceable decorator ────────────────────
# Wraps any Python function and sends trace to LangSmith

@traceable(name="retrieve_documents", run_type="retriever")
def retrieve_docs(query: str) -> list:
    """Your custom retriever function — now traced!"""
    # In production: FAISS, Pinecone, Chroma lookup
    return [f"Doc about {query}: content here...", f"Another doc on {query}"]

@traceable(name="generate_answer", run_type="llm")
def generate_answer(query: str, context: list) -> str:
    """Custom LLM call — traced with inputs/outputs"""
    # In production: direct OpenAI/Anthropic call
    return f"Answer to '{query}' based on {len(context)} docs: [Generated answer]"

@traceable(name="rag_pipeline", run_type="chain")
def rag_pipeline(query: str) -> dict:
    """Parent trace — child traces nest under this"""
    docs = retrieve_docs(query)          # child trace
    answer = generate_answer(query, docs)  # child trace
    return {"query": query, "answer": answer, "sources": docs}

# ── Method 2: Context manager ─────────────────────────
from langsmith import trace

def process_with_trace(data: str):
    with trace(name="process_data", run_type="chain", inputs={"data": data}) as run:
        result = data.upper()  # your processing
        run.end(outputs={"result": result})
    return result

# ── run_type options ──────────────────────────────────
print("run_type options for @traceable:")
run_types = {
    "llm"       : "Direct LLM calls",
    "chain"     : "Multi-step pipelines",
    "retriever" : "Document retrieval functions",
    "tool"      : "Tool/function calls in agents",
    "embedding" : "Embedding generation",
    "parser"    : "Output parsing steps",
    "prompt"    : "Prompt formatting",
}
for rtype, desc in run_types.items():
    print(f"  '{rtype:<12}' → {desc}")

In [ ]:
# ── Adding metadata to traces (production must-have) ──

from langsmith.run_helpers import langsmith_extra

@traceable
def answer_user_query(query: str, user_id: str, session_id: str) -> str:
    return f"Answer for {query}"

# Pass metadata at call time
# answer_user_query(
#     "What is LangSmith?",
#     user_id="user_rakesh_42",
#     session_id="sess_abc123",
#     langsmith_extra={
#         "metadata": {
#             "user_id": "user_rakesh_42",
#             "app_version": "v2.1.0",
#             "environment": "production",
#             "feature_flag": "new_rag_v2"
#         },
#         "tags": ["production", "rag", "v2"]
#     }
# )

# In LangChain chains, pass config:
CONFIG_EXAMPLE = '''
chain.invoke(
    {"question": "What is RAG?"},
    config={
        "run_name": "rag_query_v2",
        "tags": ["production", "rag"],
        "metadata": {
            "user_id": "user_42",
            "session_id": "sess_xyz",
            "model_version": "gpt-4o-mini"
        }
    }
)
'''
print("✅ Metadata in traces helps you:")
metadata_benefits = [
    "Filter runs by user_id to debug specific user issues",
    "Compare performance across app versions",
    "Tag prod vs dev vs staging runs",
    "Track A/B test variants",
]
for b in metadata_benefits:
    print(f"  → {b}")

---
## 15. 📊 Datasets & Evaluation

### Theory
**Datasets** are curated test cases. **Evaluation** runs your chain against them and scores outputs.

```
Dataset (input/output pairs)
         │
         ▼
evaluate(chain, data=dataset, evaluators=[...])
         │
         ▼
Scores per example → Aggregated metrics → Compare across runs
```

> 🔥 **Interview Tip**: "How do you do regression testing for LLM apps?" → Create a dataset of golden Q&A pairs, run `evaluate()` in CI/CD, fail build if score drops below threshold.


In [ ]:
from langsmith import Client

client = Client()

# ── STEP 1: Create a Dataset ──────────────────────────
dataset_name = "RAG QA Golden Dataset v1"

# Example inputs/outputs (golden test cases)
examples = [
    {
        "inputs": {"question": "What is LangSmith?"},
        "outputs": {"answer": "LangSmith is an observability platform for LLM applications"}
    },
    {
        "inputs": {"question": "What is LangServe used for?"},
        "outputs": {"answer": "LangServe deploys LangChain apps as production REST APIs"}
    },
    {
        "inputs": {"question": "How does RAG work?"},
        "outputs": {"answer": "RAG retrieves relevant documents and uses them as context for LLM generation"}
    },
]

# Create dataset (only if it doesn't exist)
CREATE_DATASET_CODE = '''
# Create dataset
dataset = client.create_dataset(
    dataset_name,
    description="Golden Q&A pairs for RAG pipeline evaluation"
)

# Add examples
client.create_examples(
    inputs=[e["inputs"] for e in examples],
    outputs=[e["outputs"] for e in examples],
    dataset_id=dataset.id
)
print(f"Dataset created: {dataset.id}")
'''
print(CREATE_DATASET_CODE)
print(f"Would create dataset '{dataset_name}' with {len(examples)} examples")

In [ ]:
from langsmith.evaluation import evaluate, LangChainStringEvaluator

# ── STEP 2: Define the chain to evaluate ─────────────
def my_rag_chain(inputs: dict) -> dict:
    """The function/chain being evaluated"""
    question = inputs["question"]
    # In production: actual RAG chain call
    answer = f"Simulated answer to: {question}"
    return {"answer": answer}

# ── STEP 3: Choose Evaluators ─────────────────────────
# Built-in LangChain evaluators
evaluators = [
    LangChainStringEvaluator("cot_qa"),          # Chain-of-thought QA correctness
    LangChainStringEvaluator("labeled_criteria", config={"criteria": "conciseness"}),
    LangChainStringEvaluator("labeled_criteria", config={"criteria": "relevance"}),
]

# ── STEP 4: Run Evaluation ────────────────────────────
RUN_EVAL_CODE = '''
results = evaluate(
    my_rag_chain,                  # function to evaluate
    data=dataset_name,             # dataset name or ID
    evaluators=evaluators,         # scoring functions
    experiment_prefix="rag-v2",    # for comparison
    metadata={"model": "gpt-4o-mini", "rag_version": "v2"}
)

# Results show:
# - Per-example scores
# - Aggregate metrics (mean, std)
# - Comparison with previous runs
print(results)
'''

print("Evaluation workflow:")
print(RUN_EVAL_CODE)

print("Built-in evaluator types:")
eval_types = {
    "qa"                  : "Is the answer correct? (uses reference answer)",
    "cot_qa"              : "QA with chain-of-thought reasoning",
    "criteria"            : "Score against custom criteria (no reference needed)",
    "labeled_criteria"    : "Score against criteria with reference answer",
    "string_distance"     : "Fuzzy string match with reference",
    "exact_match"         : "Exact string match",
    "json_validity"       : "Is output valid JSON?",
}
for name, desc in eval_types.items():
    print(f"  '{name:<22}' → {desc}")

---
## 16. 🧪 Custom Evaluators

### When built-in evaluators aren't enough
Real production needs domain-specific evaluation — e.g., "Does the answer cite a source?", "Is the answer in Hindi?", "Does the code run without errors?"


In [ ]:
from langsmith.schemas import Run, Example
from langsmith.evaluation import evaluate

# ── Method 1: Simple function evaluator ───────────────
def length_evaluator(run: Run, example: Example) -> dict:
    """
    Custom evaluator: checks if answer is a reasonable length.
    Returns score between 0 and 1.
    """
    output = run.outputs.get("answer", "")
    word_count = len(output.split())
    
    # Score: 1.0 if 20-200 words, penalize otherwise
    if 20 <= word_count <= 200:
        score = 1.0
    elif word_count < 20:
        score = word_count / 20  # too short
    else:
        score = max(0, 1 - (word_count - 200) / 200)  # too long
    
    return {
        "key": "answer_length_quality",
        "score": score,
        "comment": f"Word count: {word_count}"
    }

def source_citation_evaluator(run: Run, example: Example) -> dict:
    """Checks if answer cites sources — domain-specific quality check"""
    output = run.outputs.get("answer", "")
    has_citation = any(marker in output.lower() for marker in ["according to", "source:", "based on", "ref:"])
    return {
        "key": "has_citation",
        "score": 1 if has_citation else 0,
        "comment": "Answer includes source citation" if has_citation else "Missing source citation"
    }

# ── Method 2: LLM-as-Judge evaluator ─────────────────
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

JUDGE_PROMPT = ChatPromptTemplate.from_template("""
You are an expert evaluator. Rate the following answer on a scale of 1-5.

Question: {question}
Answer: {answer}
Reference Answer: {reference}

Rate ONLY on factual accuracy. Return ONLY a JSON: {{"score": <1-5>, "reason": "<brief reason>"}}
""")

def llm_judge_evaluator(run: Run, example: Example) -> dict:
    """Uses LLM to judge answer quality — the most flexible evaluator"""
    judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    
    try:
        response = judge_llm.invoke(
            JUDGE_PROMPT.format(
                question=example.inputs.get("question", ""),
                answer=run.outputs.get("answer", ""),
                reference=example.outputs.get("answer", "") if example.outputs else ""
            )
        )
        import json
        data = json.loads(response.content)
        return {
            "key": "llm_judge_accuracy",
            "score": data["score"] / 5,  # normalize to 0-1
            "comment": data["reason"]
        }
    except Exception as e:
        return {"key": "llm_judge_accuracy", "score": 0, "comment": f"Eval error: {e}"}

print("✅ Custom evaluators defined:")
print("  length_evaluator       → checks answer verbosity")
print("  source_citation_evaluator → domain-specific quality")
print("  llm_judge_evaluator    → LLM-as-Judge (most flexible)")
print("\n💡 In evaluate() call:")
print("  evaluators=[length_evaluator, source_citation_evaluator, llm_judge_evaluator]")

---
## 17. 👍 Feedback & Annotation

### Why Feedback Matters
Real users are your best evaluators. Capturing **thumbs up/down** from production creates a **continuous improvement loop**:

```
User gives feedback → Saved in LangSmith → Analyze low-scored runs
→ Fix prompts/retrieval → Re-evaluate on dataset → Deploy improved version
```


In [ ]:
from langsmith import Client

client = Client()

# ── Method 1: Programmatic feedback ───────────────────
FEEDBACK_CODE = '''
# After getting a response from your chain:
run_id = "the-run-id-from-trace"  # get from run metadata

# Submit feedback programmatically
client.create_feedback(
    run_id=run_id,
    key="user_rating",           # feedback dimension
    score=1,                     # 1 = thumbs up, 0 = thumbs down
    comment="Very helpful!",
    value="positive"
)

# Multi-dimensional feedback
for key, score in [("accuracy", 0.9), ("helpfulness", 0.8), ("conciseness", 0.7)]:
    client.create_feedback(run_id=run_id, key=key, score=score)
'''

# ── Method 2: LangServe feedback endpoint (auto!) ─────
LANGSERVE_FEEDBACK_CODE = '''
# If using LangServe, enable feedback endpoint:
add_routes(
    app, chain, path="/qa",
    enable_feedback_endpoint=True  # adds POST /qa/feedback
)

# Frontend calls:
# POST /qa/feedback
# Body: {"run_id": "abc", "key": "thumbs", "score": 1}
'''

# ── Method 3: Get runs with low scores for analysis ───
ANALYSIS_CODE = '''
# Find all runs with low user ratings to understand failure patterns
poor_runs = list(client.list_runs(
    project_name="my-production-app",
    filter='and(has(feedback_stats.user_rating, {"lte": 0.3}), eq(run_type, "chain"))',
    limit=50
))

for run in poor_runs:
    print(f"Input: {run.inputs}")
    print(f"Output: {run.outputs}")
    print(f"Latency: {run.end_time - run.start_time}")
    print("---")
'''

print("Feedback workflow:")
print(FEEDBACK_CODE)
print("LangServe auto-feedback:")
print(LANGSERVE_FEEDBACK_CODE)

---
## 18. 📈 Monitoring in Production

### Key Metrics to Monitor


In [ ]:
from langsmith import Client
from datetime import datetime, timedelta

client = Client()

# ── Query production traces for monitoring ────────────
MONITORING_CODE = '''
from langsmith import Client
from datetime import datetime, timedelta

client = Client()

# Get runs from last 24 hours
yesterday = datetime.utcnow() - timedelta(hours=24)

runs = list(client.list_runs(
    project_name="my-production-app",
    start_time=yesterday,
    run_type="chain",
    limit=1000
))

# ── Calculate key metrics ─────────────────────────────
total_runs = len(runs)
error_runs = [r for r in runs if r.error]
latencies = [(r.end_time - r.start_time).total_seconds() for r in runs if r.end_time]
total_tokens = sum(r.total_tokens or 0 for r in runs)

print(f"Total runs (24h):     {total_runs}")
print(f"Error rate:           {len(error_runs)/total_runs*100:.1f}%")
print(f"Avg latency:          {sum(latencies)/len(latencies):.2f}s")
print(f"P95 latency:          {sorted(latencies)[int(len(latencies)*0.95)]:.2f}s")
print(f"Total tokens used:    {total_tokens:,}")
print(f"Est. cost (gpt-4o-mini): ${total_tokens * 0.00000015:.2f}")
'''

print(MONITORING_CODE)

print("\n📊 Key production metrics to track:")
metrics = [
    ("Latency P50/P95/P99", "User experience"),
    ("Error rate",          "Reliability"),
    ("Token usage",         "Cost control"),
    ("User feedback score", "Quality"),
    ("Eval dataset score",  "Regression detection"),
    ("Retrieval hit rate",  "RAG quality"),
    ("Tool call success %", "Agent reliability"),
]
for metric, purpose in metrics:
    print(f"  {metric:<30} → {purpose}")

---
## 19. 🗄️ LangSmith Hub — Prompt Management

### Theory
LangSmith Hub is a **prompt registry**. Instead of hardcoding prompts in your code:
- Store prompts in Hub with **version control**
- **Pull** prompts at runtime — update without redeploying
- **Share** prompts across teams
- **A/B test** different prompt versions

> 🔥 **Interview Tip**: "How do you manage prompt versions in production?" — LangSmith Hub + pull at runtime = no redeploy needed for prompt updates.


In [ ]:
from langchain import hub

# ── Pull community prompts ─────────────────────────────
# These are real prompts from the LangSmith Hub!

HUB_USAGE_CODE = '''
from langchain import hub

# Pull the famous RAG prompt from Hub
rag_prompt = hub.pull("rlm/rag-prompt")        # community RAG prompt
react_prompt = hub.pull("hwchase17/react")      # ReAct agent prompt  
cot_prompt = hub.pull("langchain-ai/chain-of-thought")  # CoT reasoning

# Pull a specific version (for reproducibility)
rag_prompt_v2 = hub.pull("rlm/rag-prompt:v2")

# Use in chain — same as any prompt!
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini")
rag_chain = rag_prompt | llm | StrOutputParser()
'''

# ── Push your own prompt to Hub ───────────────────────
HUB_PUSH_CODE = '''
from langchain import hub
from langchain_core.prompts import ChatPromptTemplate

my_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Be concise and professional."),
    ("human", "{query}")
])

# Push to Hub (public or private repo)
url = hub.push("rakesh_kumar/production-qa-prompt", my_prompt)
print(f"Prompt published: {url}")

# Later, pull in production code:
prod_prompt = hub.pull("rakesh_kumar/production-qa-prompt")
# Update prompt in Hub UI → production auto-uses new version!
'''

print("Hub pull usage:")
print(HUB_USAGE_CODE)
print("Hub push (your own prompts):")
print(HUB_PUSH_CODE)
print("\nPopular Hub prompts:")
popular = [
    ("rlm/rag-prompt",          "Standard RAG Q&A prompt"),
    ("hwchase17/react",         "ReAct agent prompt"),
    ("hwchase17/openai-tools-agent", "OpenAI tools agent"),
    ("langchain-ai/sql-query-system-prompt", "Text-to-SQL"),
]
for name, desc in popular:
    print(f"  {name:<45} → {desc}")

---
---
# 🔥 PART 3: INTERVIEW Q&A + INDUSTRY PATTERNS


---
## 20. 💬 Interview Questions with Answers

---

### 🚀 LANGSERVE INTERVIEW QUESTIONS

---

#### ⭐ Q: What is LangServe and what endpoints does it auto-generate?
**A:** LangServe wraps any LangChain `Runnable` and exposes it as a FastAPI REST API. For a route at `/chat`, it auto-generates:
- `POST /chat/invoke` — sync single call
- `POST /chat/batch` — parallel multiple calls
- `POST /chat/stream` — SSE token streaming
- `POST /chat/stream_log` — intermediate steps + streaming
- `GET /chat/input_schema` — Pydantic/JSON schema
- `GET /chat/output_schema` — response schema
- `GET /chat/playground` — interactive UI

---

#### ⭐ Q: How do you add authentication to a LangServe endpoint?
**A:** Use FastAPI's `Depends` mechanism in `add_routes`:
```python
from fastapi import Depends, HTTPException
from fastapi.security import HTTPBearer

security = HTTPBearer()
def verify_token(credentials = Depends(security)):
    if credentials.credentials not in VALID_TOKENS:
        raise HTTPException(401, "Unauthorized")

add_routes(app, chain, path="/qa", dependencies=[Depends(verify_token)])
```

---

#### ⭐ Q: What is RemoteRunnable and why is it useful?
**A:** `RemoteRunnable` is LangServe's Python client. It wraps a remote LangServe URL and exposes the same `.invoke()`, `.stream()`, `.batch()`, `.ainvoke()` interface as a local chain. This enables:
1. **Microservices**: Each chain/agent as a separate service, called via RemoteRunnable
2. **Language-agnostic clients**: Any language can call the REST API
3. **Same code**: Switch between local and remote without changing calling code

---

#### ⭐ Q: What is the current status of LangServe? Should you use it in 2025?
**A:** LangServe is in **maintenance mode** — it still works but LangChain team recommends:
- **Simple chains**: LangServe is still fine
- **Agents with memory**: Use **LangGraph Platform** (replaces LangServe for agents)
- **New projects**: Start with LangGraph + LangGraph Cloud/Platform

Still important for interviews as many production systems use it.

---

#### ⭐ Q: How do you stream LangServe output to a React frontend?
**A:** Use `EventSource` (SSE) in the frontend:
```javascript
// Frontend (React)
const streamResponse = async (question) => {
  const response = await fetch('/qa/stream', {
    method: 'POST',
    headers: { 'Content-Type': 'application/json' },
    body: JSON.stringify({ input: { question } })
  });
  const reader = response.body.getReader();
  while (true) {
    const { done, value } = await reader.read();
    if (done) break;
    const chunk = new TextDecoder().decode(value);
    // parse chunk and update state
  }
};
```

---

### 🔭 LANGSMITH INTERVIEW QUESTIONS

---

#### ⭐ Q: What is LangSmith and what are its 4 core features?
**A:** LangSmith is an observability + evaluation platform for LLM apps:
1. **Tracing**: Full execution trace of every LangChain/LangGraph run — inputs, outputs, latency, tokens, errors at each step
2. **Datasets**: Versioned test case collections (input/output pairs) for evaluation
3. **Evaluation**: Run your chain against datasets with automated + LLM-as-judge scorers
4. **Hub**: Prompt version registry — store, version, pull prompts at runtime

---

#### ⭐ Q: What 4 environment variables enable LangSmith tracing?
**A:**
```bash
LANGCHAIN_TRACING_V2=true                           # Enable tracing
LANGCHAIN_ENDPOINT=https://api.smith.langchain.com  # API endpoint
LANGCHAIN_API_KEY=ls__your_key_here                 # Auth key
LANGCHAIN_PROJECT=my-app-name                       # Project namespace
```
Just setting these 4 vars traces **all** LangChain/LangGraph calls with zero code changes.

---

#### ⭐ Q: How do you trace non-LangChain code in LangSmith?
**A:** Use the `@traceable` decorator:
```python
from langsmith import traceable

@traceable(name="my_retriever", run_type="retriever")
def my_custom_retriever(query: str) -> list:
    # Any Python function — FAISS, Pinecone, SQL, etc.
    return retrieve_from_db(query)
```
Or use the `trace()` context manager for inline tracing.

---

#### ⭐ Q: How do you implement LLM regression testing with LangSmith?
**A:** Three-step process:
1. **Create Dataset**: Golden input/output pairs (`client.create_dataset` + `client.create_examples`)
2. **Run Evaluation**: `evaluate(my_chain, data=dataset, evaluators=[...])`
3. **CI/CD Integration**: Run eval in GitHub Actions, fail if score < threshold

```python
# In CI/CD pipeline
results = evaluate(chain, data="golden-dataset", evaluators=[evaluator])
avg_score = results.to_pandas()["feedback.accuracy"].mean()
assert avg_score >= 0.85, f"Quality regression! Score: {avg_score}"
```

---

#### ⭐ Q: What is LLM-as-a-Judge and when do you use it?
**A:** LLM-as-Judge uses a **separate LLM** (often GPT-4 or Claude) to evaluate the output of your chain. Use it when:
- No clear reference answer exists (open-ended generation)
- Evaluation criteria are complex (coherence, helpfulness, tone)
- Human annotation would be too slow/expensive

Caution: Judge LLM has its own biases. Use **multiple evaluators** + human spot-check.

---

#### ⭐ Q: How do you use LangSmith Hub in production? What's the advantage?
**A:**
```python
from langchain import hub
prompt = hub.pull("my-org/production-prompt:v3")  # pull at runtime
```
**Advantage**: Update the prompt in LangSmith Hub UI → production uses new version **without redeployment**. Enables:
- Hotfix broken prompts in minutes
- A/B test prompt versions
- Non-engineers can update prompts via UI
- Full version history with rollback

---

#### ⭐ Q: How do you capture user feedback in production and use it to improve your LLM app?
**A:** Continuous improvement loop:
1. **Capture**: `client.create_feedback(run_id, key="thumbs", score=0/1)` after each response
2. **Analyze**: Query LangSmith for runs with low scores — find patterns
3. **Curate**: Add failing examples to evaluation dataset
4. **Fix**: Improve prompts / retrieval / chain logic
5. **Validate**: Re-run evaluation dataset — confirm improvement
6. **Deploy**: Release new version
7. **Repeat**: Monitor in production → back to step 1


---
## 21. 🏭 Industry Patterns & System Design

---

### Pattern 1: Complete LLM App Stack

```
┌─────────────────────────────────────────────────────────────────┐
│                  PRODUCTION LLM APP ARCHITECTURE               │
│                                                                 │
│  Frontend (React/Next.js)                                       │
│       │  SSE streaming / REST                                   │
│       ▼                                                         │
│  API Gateway (nginx + rate limiting)                            │
│       │                                                         │
│       ▼                                                         │
│  LangServe (FastAPI)                                            │
│       │  add_routes for each chain/agent                        │
│       │                                                         │
│  ┌────┴─────────────────────────┐                               │
│  │                              │                               │
│  ▼                              ▼                               │
│  LangGraph Agent             RAG Chain                          │
│  (cyclic, tools, memory)     (retrieve → generate)              │
│       │                              │                          │
│       ▼                              ▼                          │
│  PostgreSQL                      FAISS / Pinecone               │
│  (checkpointing)                 (vector store)                 │
│                                                                 │
│  🔭 LangSmith                                                   │
│  (traces ALL of the above automatically)                        │
│  → Tracing | Evaluation | Feedback | Hub prompts                │
└─────────────────────────────────────────────────────────────────┘
```

---

### Pattern 2: CI/CD Pipeline for LLM Apps

```
Code Push (GitHub)
       │
       ▼
GitHub Actions CI
       │
       ├── Unit tests (node functions, chain logic)
       ├── LangSmith eval run (against golden dataset)
       │       ├── Score >= threshold? ✅ → Continue
       │       └── Score < threshold? ❌ → Fail build
       ├── Latency benchmark (<2s P95)
       └── Deploy (if all pass)
              │
              ▼
       Production LangServe
              │
              ▼
       LangSmith monitoring (ongoing)
```

---

### Pattern 3: Prompt Management Workflow

```
Prompt Engineer writes prompt
       │
       ▼
Push to LangSmith Hub (versioned)
       │
       ▼
Production code pulls at runtime:
  prompt = hub.pull("org/prompt:v2")
       │
       ├── A/B test: 50% users get v2, 50% get v3
       │       │
       │       ▼
       │   LangSmith: compare feedback scores
       │
       └── Winner → set as default version in Hub
```

---

### Pattern 4: Multi-Environment Tracing

```python
# Use different LangSmith projects per environment
import os

ENV = os.getenv("APP_ENV", "development")

PROJECT_MAP = {
    "development" : "myapp-dev",
    "staging"     : "myapp-staging",
    "production"  : "myapp-production"
}

os.environ["LANGCHAIN_PROJECT"] = PROJECT_MAP[ENV]
# Now dev/staging/prod traces are isolated in LangSmith
```


---
## 📊 Quick Reference Cheat Sheet

### LangServe

```python
# ── Core imports ──────────────────────────────────────
from fastapi import FastAPI
from langserve import add_routes, RemoteRunnable

# ── Server ────────────────────────────────────────────
app = FastAPI()
add_routes(app, my_chain, path="/chat")              # bare minimum
add_routes(app, chain, path="/chat",                 # full options
    input_type=MyInput,
    output_type=MyOutput,
    config_keys=["configurable"],
    dependencies=[Depends(verify_token)],
    enable_feedback_endpoint=True,
)

# ── Auto-generated endpoints ──────────────────────────
# POST /chat/invoke | /chat/batch | /chat/stream
# GET  /chat/playground | /chat/input_schema | /chat/output_schema

# ── Client ────────────────────────────────────────────
chain = RemoteRunnable("http://server:8000/chat")
chain.invoke(input)          # sync
chain.stream(input)          # streaming
await chain.ainvoke(input)   # async
```

### LangSmith

```python
# ── Enable tracing (env vars) ─────────────────────────
LANGCHAIN_TRACING_V2=true
LANGCHAIN_API_KEY=ls__your_key
LANGCHAIN_PROJECT=my-project

# ── Manual tracing ────────────────────────────────────
from langsmith import traceable, Client
@traceable(name="my_fn", run_type="chain")  # run_type: llm|chain|retriever|tool
def my_function(input): ...

# ── Datasets & Eval ───────────────────────────────────
client = Client()
dataset = client.create_dataset("name")
client.create_examples(inputs=[...], outputs=[...], dataset_id=dataset.id)
results = evaluate(my_chain, data="name", evaluators=[my_evaluator])

# ── Feedback ──────────────────────────────────────────
client.create_feedback(run_id, key="quality", score=0.9)

# ── Hub ───────────────────────────────────────────────
from langchain import hub
prompt = hub.pull("rlm/rag-prompt")              # pull
hub.push("my-org/my-prompt", my_prompt)          # push
```

---

## 🎯 Portfolio Project Ideas Using LangServe + LangSmith

| Project | LangServe Role | LangSmith Role | Difficulty |
|---------|----------------|----------------|------------|
| **RAG API service** | Deploy RAG chain as REST API | Trace retrievals, eval accuracy | ⭐⭐ |
| **Multi-chain API** | Multiple endpoints (QA, summarize, translate) | Compare latency across chains | ⭐⭐ |
| **Agent-as-a-service** | LangGraph agent + LangServe | Full agent trace, HITL feedback | ⭐⭐⭐ |
| **Eval-driven RAG** | Deploy, get feedback | Build dataset from prod logs, continuous eval | ⭐⭐⭐⭐ |
| **LLM Microservices** | 3+ RemoteRunnable services | Distributed trace across services | ⭐⭐⭐⭐ |

---

*Notebook by Rakesh Kumar | LangServe + LangSmith Revision | Production-Grade*
